# N2UQ Quantized ResNet Inference

Load and run inference with Nonuniform-to-Uniform Quantization (N2UQ) pretrained ResNet models.

Reference: [Nonuniform-to-Uniform Quantization: Towards Accurate Quantization via Generalized Straight-Through Estimation](https://arxiv.org/abs/2111.14826) (CVPR 2022)

In [ ]:
import os
import sys
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# Add project root to path so we can import the resnet module
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from resnet import resnet18, resnet34, resnet50

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
# Model configuration
ARCH = "resnet18"        # resnet18, resnet34, resnet50
N_BITS = 2               # 2, 3, or 4
QUANTIZE_DOWNSAMPLE = True

# Paths
MODEL_DIR = "pretrained_models"

# Select checkpoint file
MODEL_FILES = {
    (True, "resnet18"): "quantize_downsampling_true/real_res18.pth.tar",
    (True, "resnet34"): "quantize_downsampling_true/real_res34.pth.tar",
    (True, "resnet50"): "quantize_downsampling_true/real_res50.pth.tar",
    (False, "resnet18"): "quantize_downsampling_false/resnet18-f37072fd(pytorch_model).pth",
    (False, "resnet34"): "quantize_downsampling_false/resnet34-b627a593(pytorch_model).pth",
    (False, "resnet50"): "quantize_downsampling_false/resnet50-0676ba61(pytorch_model).pth",
}

checkpoint_path = os.path.join(MODEL_DIR, MODEL_FILES[(QUANTIZE_DOWNSAMPLE, ARCH)])
print(f"Checkpoint: {checkpoint_path}")
print(f"File exists: {os.path.exists(checkpoint_path)}")

## Load the N2UQ model

In [ ]:
# Instantiate model
MODEL_BUILDERS = {
    "resnet18": resnet18,
    "resnet34": resnet34,
    "resnet50": resnet50,
}

model = MODEL_BUILDERS[ARCH](n_bit=N_BITS, quantize_downsample=QUANTIZE_DOWNSAMPLE)

# Load pretrained weights
checkpoint = torch.load(checkpoint_path, map_location="cpu")
if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

model.load_state_dict(state_dict, strict=False)
model.eval()

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Model: {ARCH} ({N_BITS}-bit, quantize_downsample={QUANTIZE_DOWNSAMPLE})")
print(f"Device: {device}")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## ImageNet preprocessing

In [ ]:
# Standard ImageNet normalization
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Load ImageNet class labels
# If no labels file is available, use index-based labels
LABELS_PATH = "imagenet_classes.txt"
if os.path.exists(LABELS_PATH):
    with open(LABELS_PATH) as f:
        imagenet_labels = [line.strip() for line in f.readlines()]
else:
    imagenet_labels = None
    print("No imagenet_classes.txt found; will show class indices.")

## Inference helper

In [ ]:
def predict(image_path, model, preprocess, device, topk=5):
    """Run inference on a single image."""
    img = Image.open(image_path).convert("RGB")
    input_tensor = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.nn.functional.softmax(output[0], dim=0)

    top_probs, top_indices = torch.topk(probs, topk)

    # Display
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.imshow(img)
    ax1.set_title("Input Image")
    ax1.axis("off")

    labels = []
    probs_list = []
    for i in range(topk):
        idx = top_indices[i].item()
        label = imagenet_labels[idx] if imagenet_labels else f"class {idx}"
        labels.append(f"{label}: {top_probs[i].item()*100:.1f}%")
        probs_list.append(top_probs[i].item())

    ax2.barh(range(topk), probs_list[::-1])
    ax2.set_yticks(range(topk))
    ax2.set_yticklabels(labels[::-1])
    ax2.set_xlabel("Probability")
    ax2.set_title(f"Top-{topk} Predictions")
    plt.tight_layout()
    plt.show()

    return top_indices, top_probs

## Run inference

Place an image in this directory and update `image_path` below, or use a sample from ImageNet validation.

In [ ]:
# Example: random input to verify the model runs
dummy_input = torch.randn(1, 3, 224, 224, device=device)
with torch.no_grad():
    output = model(dummy_input)
print(f"Output shape: {output.shape}")
print(f"Top-5 class indices: {torch.topk(output, 5).indices.tolist()}")
print("Model is ready for inference.")

In [ ]:
# Uncomment and set image_path to run on a real image:
# image_path = "path/to/your/image.jpg"
# top_indices, top_probs = predict(image_path, model, preprocess, device)